In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import copy

# Paths
DATASET_PATH = "your path"
RESULT_PATH = "your path"
os.makedirs(RESULT_PATH, exist_ok=True)

# Load dataset
df = pd.read_csv(DATASET_PATH)

# Preprocess date
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')

# Sort by Company and Date
df = df.sort_values(['Company', 'Date']).reset_index(drop=True)

# Normalize features: Open, High, Low, Close, Adj Close, Volume
features = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
scalers = {}
for feat in features:
    scaler = StandardScaler()
    df[feat] = scaler.fit_transform(df[[feat]])
    scalers[feat] = scaler

# Define prediction target: Next day Close price
df['Target_Close'] = df.groupby('Company')['Close'].shift(-1)
df = df.dropna(subset=['Target_Close']).reset_index(drop=True)

# Create graphs: one graph per company per month
# Nodes: days in that month, features per day
# Edges: connect consecutive days bidirectionally
# Target per graph: last day's Target_Close value (prediction target)

def create_graphs(df):
    graphs = []
    df['YearMonth'] = df['Date'].dt.to_period('M')
    grouped = df.groupby(['Company', 'YearMonth'])

    for (company, ym), group in grouped:
        group = group.sort_values('Date').reset_index(drop=True)
        if len(group) < 2:
            continue
        x = torch.tensor(group[features].values, dtype=torch.float)
        # Target: Target_Close of last node in graph (next day close of last day)
        y = torch.tensor([group['Target_Close'].values[-1]], dtype=torch.float).view(-1, 1)

        edge_index = []
        for i in range(len(group) - 1):
            edge_index.append([i, i+1])
            edge_index.append([i+1, i])
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

        data = Data(x=x, edge_index=edge_index, y=y)
        data.company = company
        data.year_month = str(ym)
        graphs.append(data)
    return graphs

graphs = create_graphs(df)
print(f"Total graphs created: {len(graphs)}")

# Split graphs by company (simulate FL clients)
clients = {}
for g in graphs:
    clients.setdefault(g.company, []).append(g)

print(f"Clients: {list(clients.keys())}")

# Define GCN Regression model
class GCNRegression(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(GCNRegression, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, 1)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = global_mean_pool(x, batch)  # Aggregate node embeddings per graph
        x = self.lin(x)
        return x

# Training function for one client
def train(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.batch.to(device))
        loss = F.mse_loss(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

# Evaluation function
def evaluate(model, loader, device):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.batch.to(device))
            preds.append(out.cpu())
            trues.append(data.y.cpu())
    preds = torch.cat(preds).view(-1)
    trues = torch.cat(trues).view(-1)
    mse = mean_squared_error(trues, preds)
    return mse, preds, trues

# Prepare dataloaders per client (80% train, 20% test split)
from torch_geometric.loader import DataLoader

client_loaders = {}
for c, g_list in clients.items():
    n = len(g_list)
    train_n = int(0.8 * n)
    train_loader = DataLoader(g_list[:train_n], batch_size=4, shuffle=True)
    test_loader = DataLoader(g_list[train_n:], batch_size=4, shuffle=False)
    client_loaders[c] = (train_loader, test_loader)

# Federated Learning Simulation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
in_channels = len(features)
hidden_channels = 64

def get_model():
    model = GCNRegression(in_channels, hidden_channels)
    return model.to(device)

def average_weights(w_list):
    avg_w = copy.deepcopy(w_list[0])
    for key in avg_w.keys():
        for w in w_list[1:]:
            avg_w[key] += w[key]
        avg_w[key] = torch.div(avg_w[key], len(w_list))
    return avg_w

epochs_per_round = 3
rounds = 10

global_model = get_model()
global_weights = global_model.state_dict()

mse_per_round = []

for r in range(rounds):
    print(f"Round {r+1}/{rounds}")
    local_weights = []
    for c, (train_loader, _) in client_loaders.items():
        local_model = get_model()
        local_model.load_state_dict(global_weights)
        optimizer = torch.optim.Adam(local_model.parameters(), lr=0.01)
        for e in range(epochs_per_round):
            train(local_model, train_loader, optimizer, device)
        local_weights.append(local_model.state_dict())
    global_weights = average_weights(local_weights)
    global_model.load_state_dict(global_weights)

    # Evaluate on combined test set of all clients
    all_test_data = []
    for _, (_, test_loader) in client_loaders.items():
        all_test_data += test_loader.dataset
    all_test_loader = DataLoader(all_test_data, batch_size=8, shuffle=False)
    mse, preds, trues = evaluate(global_model, all_test_loader, device)
    print(f"Global Model MSE: {mse:.4f}")
    mse_per_round.append(mse)

# Save results and plot
plt.figure()
plt.plot(range(1, rounds+1), mse_per_round, marker='o')
plt.title("Global Model MSE per FL Round")
plt.xlabel("Round")
plt.ylabel("MSE")
plt.grid(True)
plt.savefig(os.path.join(RESULT_PATH, "mse_per_round.png"))
plt.close()

with open(os.path.join(RESULT_PATH, "mse_per_round.txt"), "w") as f:
    for i, val in enumerate(mse_per_round, 1):
        f.write(f"Round {i}: MSE={val}\n")

print(f"Training finished. Results saved in {RESULT_PATH}")


Total graphs created: 2636
Clients: ['DELL', 'IBM', 'INTC', 'MSFT', 'SONY', 'VZ']
Round 1/10
Global Model MSE: 1.3876
Round 2/10
Global Model MSE: 0.7084
Round 3/10
Global Model MSE: 0.5687
Round 4/10
Global Model MSE: 0.0595
Round 5/10
Global Model MSE: 0.3357
Round 6/10
Global Model MSE: 0.0965
Round 7/10
Global Model MSE: 0.0902
Round 8/10
Global Model MSE: 0.0408
Round 9/10
Global Model MSE: 0.1661
Round 10/10
Global Model MSE: 0.0903
Training finished. Results saved in /Volumes/SP_SAGHAR/Documents/University/Articles/Conference/6- ICAEA -sbu - enterprise/result/gcn+fl
